# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Week 6 ended on a verdict: with these five features, the decline-review queue is a **one-month instrument** — refit monthly, use as decision-support, and expect the head-of-queue lift to shrink out of time. This week I stop adding model and turn that verdict into the thing it was always for: a **content action playbook** — one notebook, run once per month, that

1. grades last cycle's shipping cadence honestly against the one outcome month I have never touched (June 2026, sealed since the data was cut),
2. refits on the freshest labeled frame and scores the new month into a **ranked queue** where every row says *what to do, why (reason codes), what a human must check first, and roughly what it costs*, and
3. writes the exact files the paper's recommendations section builds on: the queue CSV (out of git by design), a receipt JSON, and four figures.

The deliverable is a plan a person can run, not a model that acts. House rules from Week 6 apply throughout: **observed / measured / directional / decision-support** — and no number without its receipt.

> Sections follow the card: §1 ranked actions + reason codes (with the archetype→action map and the decay/refresh insight) → §2 intended use, cost/value, limits → §3 human review + the no-go list → §4 monitoring / retrain triggers → §5 exports for the paper → self-check.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Decisions fixed before any new number exists

Four shipping decisions, pre-registered from Week-6 evidence — then the cells below measure, build, and export. Nothing downstream gets to revise them after seeing a result.

1. **Cadence — a July 2026 queue, built the way Week 6 prescribed.** Refit on the freshest fully-labeled frame (May 2026 features → June 2026 labels), score June 2026 features, speak for July only. Week 6's walk-forward saw its single best out-of-time draw with two frames of history (LR precision@50 0.74 on April), but the head slid back down with three and four frames — one lucky origin is not a cadence. I ship the simplest cadence with the most evidence: **one freshest frame, one month of validity**. The two-frame variant is written into the §4 runbook as the first thing to try if the head-value trigger fires.

2. **Scorer — logistic regression.** Pre-registered from the out-of-time receipts, not from this week's test: LR matched or beat the random forest at precision@50 in **all five out-of-time draws Week 6 ran** (four walk-forward origins: 0.54 / 0.74 / 0.56 / 0.50 vs RF's 0.26 / 0.50 / 0.46 / 0.48, plus the March→April draw where both read 0.54), and it carried the better Brier score (0.240 vs 0.252) — probabilities that mean something when an editor reads "risk 0.7". RF stays in this notebook as the within-month champion (grouped-fold precision@50 0.740 vs LR's 0.640) that did not survive the month. **The sealed backtest below measures the shipping cadence; it does not select the model.** LR ships July regardless of the draw, and if the draw is bad, §4's triggers — not a re-pick — say what happens next. That ordering is the honesty hinge of this notebook.

3. **Ranking key.** LR probability descending; ties broken by bigger traffic first, then IDs — determinism on purpose. Week 4 taught the cost of tie bands: a 24,775-page score-1.00 band is not a ranking, it is a lottery.

4. **Risk tiers.** `high` ≥ 0.65, `elevated` ≥ 0.55, else `background` — round cuts fixed *before* the calibration table exists. The table then grades the cuts; miscalibration is a §4 trigger, not a reason to move the cuts after the fact.

### Reason codes — every code names its evidence

| code | a human reads | where it comes from |
|---|---|---|
| `model_risk_high` / `model_risk_elevated` | the refit model puts next-30-day decline risk ≥ 0.65 / ≥ 0.55 | this notebook's ship fit (May→June) |
| `peak_risk_age_91_180` | age 91–180d — the band where measured decline peaked (0.615, March frame) | Week-4 audit |
| `stale_91d_plus` | 91+ days old — the staleness signal (0.550 vs 0.425 decline) | Week-4 audit |
| `low_ctr_top10` | visible, top-10 position, CTR under 1% — the rule's core profile | Week-4 rule |
| `top3_zero_click_suspected` | top-3 position, CTR under 0.30% — measurement or zero-click SERP suspicion, **not** decay | Week-5 misses (all three: one client, pos ≤ 2, CTR 0.07–0.18%); Week-4 top-20 |
| `thin_data_100_499` | 100–499 impressions — CTR and position estimates too noisy to act on | Week-4/5 visibility-floor note |
| `big_traffic_at_stake_5k` | ≥ 5,000 impressions/30d — worth senior attention | §2 cost/value |
| `threshold_edge` | within 10% of a decision cut (500 impressions / 1.0% CTR / 91 days) — numbers here are unstable | Week-4 weak picks |
| `client_cluster_top50` | this client holds ≥ 2× its frame share inside the top 50 — escalate to client level | Week-4 (4 of the top 10 from one client); Week-5 (all 3 misses from one client) |
| `model_risk_background` | below 0.55 — on the list, not the head | — |

The Week-4 rule's own codes ride along verbatim in `rule_reason_code`: the rule is the fallback ranking (§4), and the two vocabularies stay legible side by side in the export.

### Archetype → action mapping — cheapest check first

| archetype (in precedence order) | action | a person reads |
|---|---|---|
| `top3_zero_click_suspected` | `investigate_measurement_first` | verify tracking / zero-click SERP features before touching content |
| `thin_data_100_499` | `monitor_thin_data` | too thin to act; revisit when volume grows |
| `stale_top10_low_ctr` | `refresh_and_review_ctr` | old + visible + top-10 + CTR < 1%: refresh content, review title/intent |
| `stale_visible` | `refresh_review` | old + visible: decay-window review |
| `young_top10_low_ctr` | `review_intent_first` | under 91d: may still be settling; check query intent before spending a refresh |
| `visible_healthy` | `monitor` | ranked by model risk only; no editorial flag |

Precedence is a rule, not an accident: when two stories fit the same row, the cheap check wins — a tracking check costs minutes, a rewrite costs an hour, and thin data undermines any CTR story, so measurement suspicion and thin data both sit above the editorial profiles.

### The decay/refresh insight, stated at its honest size

Two facts and one non-fact. **Fact 1 (measured, March frame):** decline rate by age band peaks at 0.615 in the 91–180-day band and falls to 0.478 at 365+ days — decay is observable in this portfolio, directionally. **Fact 2 (measured this week):** the same curve re-measured on the May frame below — if the peak survives outside the development month, the `peak_risk_age_91_180` band keeps its basis; if not, it is a March artifact and §4's signal-liveness trigger fires. **The non-fact:** the paper's freshness multiplier — refreshed 365+ pages "gained 3.2× health, 57× impressions" — is selection, not treatment: pages with proven visibility get refreshed (Week 6, Finding A). So this playbook's refresh actions claim **review priority, never recovery** — no sentence anywhere says refreshing will regain impressions. The one honest upgrade path is §3's action log (page, action, date for every acted-on row): enough logged refreshes eventually allow a matched refreshed-vs-not comparison — the design that could earn the word "causes". Until then, directional only.

The cells below, in order: the machinery (Week-6 frame builder verbatim, plus a label-free scoring variant), the sealed-month backtest that grades the cadence, the decay receipts on a fresh month, and the build of the July queue itself.


In [ ]:
# Setup: Week-6 machinery, verbatim where possible, plus one new piece - a
# label-free scoring-frame builder (July's outcomes do not exist yet; the queue
# must not pretend otherwise). This notebook IS the monthly refit cadence Week 6
# prescribed, run once for the July 2026 queue. The month constants below are
# the only edits a future cycle needs.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

import os
import getpass
import json
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import duckdb

SEED = 42  # every stochastic step in this notebook uses this one seed

print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)

# --- this cycle's three months (the only edits next cycle needs) ---
BACKTEST_TRAIN_MONTH = "month=2026-04"  # trains the sealed backtest (freshest labeled frame at May month-end)
SHIP_TRAIN_MONTH = "month=2026-05"      # trains the shipped model (freshest labeled frame at June month-end)
SCORE_MONTH = "month=2026-06"           # features scored into the July queue
QUEUE_FOR = "2026-07"

# --- token resolution: env var -> Colab secret -> repo .env -> interactive prompt ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and Path("../../.env").exists():
    for line in Path("../../.env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")


def _months_between(start, end):
    """Calendar months (YYYY-MM) covering [start, end], inclusive."""
    months = []
    y, m = start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return months


def build_frame(feature_month: str):
    """Week-6 frame SQL, verbatim: recent-window features from the FEATURE
    month, label = >=20% impressions drop over the next 30 days, inner-joined
    to the future so every row is 'measurable next month'."""
    fact_feat = f"read_parquet('{REL}/fact_content_daily_performance/{feature_month}/*.parquet')"

    cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {fact_feat}").fetchone()[0]
    label_start = cutoff_date + timedelta(days=1)
    label_end = cutoff_date + timedelta(days=30)

    fut_months = _months_between(label_start, label_end)
    fut_paths = ", ".join(
        f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in fut_months
    )
    fact_fut = f"read_parquet([{fut_paths}])"

    part_min, part_max = con.sql(
        f"SELECT MIN(report_date), MAX(report_date) FROM {fact_fut}"
    ).fetchone()
    print(f"Feature month {feature_month} | cutoff {cutoff_date} | "
          f"label window {label_start} .. {label_end} | "
          f"label partitions {fut_months} | bounds {part_min} .. {part_max}")
    assert str(part_min) == str(label_start), "Label partitions start late - label window uncovered"
    assert part_max >= label_end, "Label partitions end before the label window - label window uncovered"

    frame = con.sql(f"""
        WITH recent AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS recent30_impressions,
                SUM(gsc_clicks) AS recent30_clicks,
                AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
                COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
                COUNT(DISTINCT report_date) AS recent30_days
            FROM {fact_feat}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        future AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS future30_impressions,
                COUNT(DISTINCT report_date) AS future30_days
            FROM {fact_fut}
            WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT
            r.client_hash_id,
            r.content_hash_id,
            r.recent30_impressions,
            LN(1 + r.recent30_impressions) AS log_recent30_impressions,
            100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
            r.recent30_avg_position,
            r.recent30_active_days,
            DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
            f.future30_impressions,
            CASE
                WHEN r.recent30_impressions >= 100
                     AND f.future30_impressions < 0.80 * r.recent30_impressions
                THEN 1 ELSE 0
            END AS is_declining_next30
        FROM recent r
        INNER JOIN future f USING (client_hash_id, content_hash_id)
        LEFT JOIN (
            SELECT client_hash_id, content_hash_id, content_created_date
            FROM {DIM_CONTENT}
        ) c USING (client_hash_id, content_hash_id)
        WHERE r.recent30_days >= 14
          AND f.future30_days >= 14
          AND r.recent30_impressions >= 100
    """).df()

    # Stable row order => deterministic ranking metrics (SQL GROUP BY order is not stable).
    frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

    meta = {
        "feature_month": feature_month,
        "future_partitions": fut_months,
        "cutoff": str(cutoff_date),
        "label_window": [str(label_start), str(label_end)],
        "rows": int(len(frame)),
        "base_rate": float(frame["is_declining_next30"].mean()),
        "clients": int(frame["client_hash_id"].nunique()),
        "largest_client_share": float(frame["client_hash_id"].value_counts(normalize=True).iloc[0]),
    }
    return frame, meta


def build_scoring_frame(feature_month: str):
    """New this week: the same recent-window features and recent-side filters,
    with NO future join and NO label - the queue is built for a month whose
    outcomes do not exist yet. Eligibility is therefore 'measurable now',
    slightly wider than the labeled frames' 'measurable next month' (the
    ~2.8-3.6% survivorship gap Week 6 measured). Section 2 states that
    boundary with this cycle's counts."""
    fact_feat = f"read_parquet('{REL}/fact_content_daily_performance/{feature_month}/*.parquet')"
    cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {fact_feat}").fetchone()[0]

    frame = con.sql(f"""
        WITH recent AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS recent30_impressions,
                SUM(gsc_clicks) AS recent30_clicks,
                AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
                COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
                COUNT(DISTINCT report_date) AS recent30_days
            FROM {fact_feat}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT
            r.client_hash_id,
            r.content_hash_id,
            r.recent30_impressions,
            LN(1 + r.recent30_impressions) AS log_recent30_impressions,
            100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
            r.recent30_avg_position,
            r.recent30_active_days,
            DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days
        FROM recent r
        LEFT JOIN (
            SELECT client_hash_id, content_hash_id, content_created_date
            FROM {DIM_CONTENT}
        ) c USING (client_hash_id, content_hash_id)
        WHERE r.recent30_days >= 14
          AND r.recent30_impressions >= 100
    """).df()

    frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
    meta = {
        "feature_month": feature_month,
        "cutoff": str(cutoff_date),
        "queue_for_window": [str(cutoff_date + timedelta(days=1)), str(cutoff_date + timedelta(days=30))],
        "rows": int(len(frame)),
        "clients": int(frame["client_hash_id"].nunique()),
    }
    return frame, meta


df_april, meta_april = build_frame(BACKTEST_TRAIN_MONTH)
df_may, meta_may = build_frame(SHIP_TRAIN_MONTH)
df_june, meta_june = build_scoring_frame(SCORE_MONTH)

# Receipts: the April frame must reproduce Week 6's committed numbers exactly.
assert len(df_april) == 99_279, ("April frame != Week-6 receipt (99,279 rows) - "
                                 "investigate before trusting anything below")
assert meta_april["cutoff"] == "2026-04-30"
assert abs(meta_april["base_rate"] - 0.5596148228729138) < 1e-8, "April base rate drifted from the Week-6 receipt"
assert meta_may["label_window"] == ["2026-06-01", "2026-06-30"], "May-frame label window unexpected"
assert meta_june["cutoff"] == "2026-06-30", "June partition ends early - scoring frame incomplete"
print("April-frame receipts vs Week 6: PASS (99,279 rows, cutoff 2026-04-30, base rate "
      f"{meta_april['base_rate']:.4f} vs receipt 0.5596)")
print(f"May frame (new this week; its labels are June outcomes - the sealed month's first touch): "
      f"{meta_may['rows']:,} rows, base rate {meta_may['base_rate']:.4f}, {meta_may['clients']} clients")
print(f"June scoring frame (new): {meta_june['rows']:,} rows, {meta_june['clients']} clients, "
      f"cutoff {meta_june['cutoff']}, queue for {meta_june['queue_for_window'][0]}..{meta_june['queue_for_window'][1]}")

TARGET = "is_declining_next30"
FEATURES = [
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
]

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    ),
}


def make_baseline_scores(frame):
    """Verbatim Week-4 hand rule. No fitted parameters. The fallback order."""
    is_visible = (frame["recent30_impressions"] >= 500).astype(int)
    is_top10 = ((frame["recent30_avg_position"] > 0) & (frame["recent30_avg_position"] <= 10)).astype(int)
    is_low_ctr = (frame["recent30_ctr_pct"] < 1.0).astype(int)
    is_stale = (frame["content_age_days"] >= 91).astype(int)
    low_ctr_top10 = is_visible * is_top10 * is_low_ctr
    visible_stale = is_visible * is_stale
    return 0.40 * is_visible + 0.35 * low_ctr_top10 + 0.25 * visible_stale


def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)


# Week-6 receipts cited in the prose - transcribed here, verified against the
# committed file whenever the notebook runs inside a repo checkout.
W6_PATH = Path("../../work/outputs/validation_audit_metrics.json")
W6 = {
    "grouped_cv_p50_means": {"baseline": 0.512, "logistic_regression": 0.640, "random_forest": 0.740},
    "grouped_cv_base_rate": 0.510,
    "time_forward_base_rate": 0.5596,
    "time_forward": {"logistic_regression": {"p50": 0.54, "ap": 0.626, "brier": 0.240},
                     "random_forest": {"p50": 0.54, "ap": 0.588, "brier": 0.252}},
    "walk_forward_p50_by_origin": {"logistic_regression": [0.54, 0.74, 0.56, 0.50],
                                   "random_forest": [0.26, 0.50, 0.46, 0.48]},
    "survivorship_dropped_share": {"march": 0.0276, "april": 0.0356},
    "rule_tie_band": {"n": 24775, "decline_rate": 0.532},
}
if W6_PATH.exists():
    r = json.loads(W6_PATH.read_text())
    assert abs(r["grouped_cv"]["summary"]["logistic_regression"]["p50_mean"] - 0.640) < 0.001
    assert abs(r["grouped_cv"]["summary"]["random_forest"]["p50_mean"] - 0.740) < 0.001
    assert abs(r["time_forward"]["metrics"]["logistic_regression"]["brier"] - 0.240) < 0.001
    print("Week-6 receipt file found; transcribed numbers verified: PASS")
else:
    print("Week-6 receipt file not present (plain Colab run) - citing transcribed numbers;")
    print("they trace to the committed work/outputs/validation_audit_metrics.json.")

print("\nSetup complete: 3 frames, contracted features, models, hand rule, metrics, receipts.")


**Pre-registered reading of the backtest** — written before the cell below ran:

- The sealed month (June 2026 outcomes, first touch since the data was cut) grades the *exact* shipping cadence: train on the freshest labeled frame, score the next month's features. One draw, no tuning, recorded as-is.
- **If LR's precision@50 clears the May-frame base rate by ≥ 5 points**, the July queue's order carries measured — one-draw, directional — support at the head.
- **If it lands at or below base**, the queue still ships: every row is decision-support, and its reason codes carry non-model evidence that stands on its own. But the shipped note must say plainly that the head did not separate from base rate this cycle, and the §4 head-value trigger is armed — a second consecutive miss ships next month's queue rule-ordered until the model re-earns the slot with one clean month.
- The rule's row prices the fallback on the same month. RF's row is reported because it ran — not because it gets to win anything after the fact.


In [ ]:
# The sealed-month backtest: the shipping cadence, measured once, on the one
# outcome month never touched before (June 2026 - sealed since the data was
# cut; the May frame's labels are its first and only use). At May month-end the
# freshest labeled frame is April (Apr features -> May labels); the production
# act is to refit on it and score May features. The May frame's labels then
# grade that queue. Pre-registered above: LR ships July regardless of this
# draw; tier cuts are already fixed; everything below is recorded as-is.
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

train = df_april.dropna(subset=FEATURES).copy()
test = df_may.dropna(subset=FEATURES).copy()
y_test = test[TARGET].to_numpy()
may_base = float(y_test.mean())
print(f"Backtest train: April frame, {len(train):,} rows (base rate {train[TARGET].mean():.4f})")
print(f"Backtest test:  May frame, {len(test):,} rows (base rate {may_base:.4f}) - June outcomes, first touch")

scorer_preds = {}
for name in ["logistic_regression", "random_forest"]:
    m = clone(models[name])
    m.fit(train[FEATURES], train[TARGET])
    scorer_preds[name] = m.predict_proba(test[FEATURES])[:, 1]
scorer_preds["hand_rule"] = make_baseline_scores(test).to_numpy()

rows = []
for name, preds in scorer_preds.items():
    rows.append({
        "scorer": name,
        "p20": precision_at_k(y_test, preds, 20),
        "p50": precision_at_k(y_test, preds, 50),
        "ndcg50": ndcg_at_k(y_test, preds, 50),
        "ap": average_precision_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, preds),
        "brier": brier_score_loss(y_test, preds),  # rough for the rule: scores, not probabilities
    })
backtest = pd.DataFrame(rows)
print("\nSealed-month backtest (train April frame -> score May features -> June outcomes):")
print(backtest.round(4).to_string(index=False))
print(f"Read every precision against this base rate: {may_base:.4f} - a precision without its base is not a claim.")

# Calibration of the shipping scorer (bins fixed before looking - Section 1's tier cuts):
preds_lr = scorer_preds["logistic_regression"]
bin_labels = ["<0.40", "0.40-0.50", "0.50-0.60", "0.60-0.70", ">=0.70"]
idx = np.digitize(preds_lr, [0.40, 0.50, 0.60, 0.70])
calib_rows = []
for i, lab in enumerate(bin_labels):
    mask = idx == i
    n = int(mask.sum())
    calib_rows.append({
        "prob_bin": lab, "n": n,
        "mean_pred": float(preds_lr[mask].mean()) if n else float("nan"),
        "realized_decline_rate": float(y_test[mask].mean()) if n else float("nan"),
    })
calib = pd.DataFrame(calib_rows)
print("\nCalibration of the shipping scorer on the sealed month:")
print(calib.round(4).to_string(index=False))
print("Reading: 'risk 0.70' is worth exactly what the >=0.70 row's realized rate says - no more.")


In [ ]:
# Decay receipts, measured on the fresh May frame (not the March development
# month): the age curve that grounds the archetypes, and the decline magnitude
# that grounds the exposure column. Both feed Section 4's liveness checks.
age_bands = pd.cut(df_may["content_age_days"], [-np.inf, 90, 180, 270, 365, np.inf],
                   labels=["<91", "91-180", "181-270", "271-365", "366+"])
age_curve = df_may.groupby(age_bands, observed=False)[TARGET].agg(["count", "mean"])
age_curve.index = age_curve.index.astype(str)

print("Decline rate by content age - May frame (fresh month, not the March dev frame):")
for band, row in age_curve.iterrows():
    print(f"  {band:>7}: {row['mean']:.3f}  (n={int(row['count']):,})")
print("Week-4 March-frame audit for comparison: peak 0.615 at 91-180d, 0.478 at 366+.")
print("Reading rule (fixed in Section 1): a peak inside 91-180 keeps the archetype basis;")
print("a flat curve demotes peak_risk_age_91_180 to noise and arms the Section-4 watch.")

decl = df_may[df_may[TARGET] == 1]
drop_share = 1.0 - decl["future30_impressions"] / decl["recent30_impressions"]
print(f"\nMeasured drop among May-frame decliners (n={len(drop_share):,}): "
      f"median {drop_share.median():.1%}, mean {drop_share.mean():.1%}")
print("(The label floor is the 20% threshold by definition; the median is what the exposure")
print("column uses - robust to the heavy tail of near-total drops.)")


In [ ]:
# The shipping act: refit on the freshest fully-labeled frame (May features ->
# June labels), score June features, and attach the playbook layer - flags,
# archetypes, actions, reason codes, tiers, exposure. Nothing here touches
# July: July has no data yet, and the queue is a ranked hypothesis list for it.
train_may = df_may.dropna(subset=FEATURES).copy()
ship = clone(models["logistic_regression"])
ship.fit(train_may[FEATURES], train_may[TARGET])

q = df_june.dropna(subset=FEATURES).copy()
print(f"Scoring frame: {len(df_june):,} June rows -> {len(q):,} scored "
      f"({len(df_june) - len(q):,} dropped for missing position/age data - disclosed, not hidden)")
q["lr_prob"] = ship.predict_proba(q[FEATURES])[:, 1]

# Ranking key (Section 1): probability desc, bigger traffic first, then IDs - determinism.
q = q.sort_values(["lr_prob", "recent30_impressions", "client_hash_id", "content_hash_id"],
                  ascending=[False, False, True, True]).reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

# ---- rule layer (Week-4 flags, verbatim logic) ----
q["rule_score"] = make_baseline_scores(q)
imp, ctr = q["recent30_impressions"], q["recent30_ctr_pct"]
pos, age = q["recent30_avg_position"], q["content_age_days"]
is_visible = imp >= 500
is_top10 = (pos > 0) & (pos <= 10)
is_low_ctr = ctr < 1.0
is_stale = age >= 91
is_peak_age = is_stale & (age <= 180)
is_thin = imp < 500
is_top3_zero_click = (pos > 0) & (pos <= 3) & (ctr < 0.30)
has_low_ctr_top10 = is_visible & is_top10 & is_low_ctr
q["rule_reason_code"] = np.select(
    [(is_visible & is_stale) & has_low_ctr_top10, is_visible & is_stale,
     has_low_ctr_top10, is_visible],
    ["stale_low_ctr_top10", "stale", "low_ctr_top10", "visible"],
    default="low_visibility",
)

# ---- cluster flag (base-rate aware): a client is a cluster when it holds >= 2x
# its frame share of the top 50. The largest client holds ~22% of frame rows, so
# ~11 of 50 is expectation, not suspicion.
top50 = q.head(50)
frame_share = q["client_hash_id"].value_counts(normalize=True)
t50_counts = top50["client_hash_id"].value_counts()
cluster_clients = sorted(c for c, n in t50_counts.items()
                         if (n / 50.0) >= 2.0 * float(frame_share.get(c, 0.0)))
q["client_cluster_top50"] = (q["rank"] <= 50) & q["client_hash_id"].isin(cluster_clients)

q["threshold_edge"] = imp.between(450, 550) | ctr.between(0.9, 1.1) | age.between(82, 100)

# ---- archetype -> action (cheapest check first; the Section 1 table) ----
q["archetype"] = np.select(
    [is_top3_zero_click,
     is_thin,
     is_visible & is_stale & is_top10 & is_low_ctr,
     is_visible & is_stale,
     is_visible & is_top10 & is_low_ctr,
     is_visible],
    ["top3_zero_click_suspected", "thin_data_100_499", "stale_top10_low_ctr",
     "stale_visible", "young_top10_low_ctr", "visible_healthy"],
    default="low_visibility",
)

ACTION_BY_ARCHETYPE = {
    "top3_zero_click_suspected": "investigate_measurement_first",
    "thin_data_100_499": "monitor_thin_data",
    "stale_top10_low_ctr": "refresh_and_review_ctr",
    "stale_visible": "refresh_review",
    "young_top10_low_ctr": "review_intent_first",
    "visible_healthy": "monitor",
    "low_visibility": "monitor",
}
REVIEW_NOTE_BY_ARCHETYPE = {
    "top3_zero_click_suspected": "Top-3 position with CTR under 0.30%: verify tracking and zero-click SERP features BEFORE any rewrite - the pattern behind Week-5's confident misses.",
    "thin_data_100_499": "100-499 impressions: CTR and position estimates are noisy; revisit when volume grows.",
    "stale_top10_low_ctr": "Old, visible, top-10, CTR under 1%: the audit's highest-priority profile; refresh content and review title/intent against the query mix.",
    "stale_visible": "Old and still visible: inside the measured decay window; review for factual decay, broken links, outdated examples.",
    "young_top10_low_ctr": "Top-10 with CTR under 1% but under 91 days old: may still be settling; check query intent before spending a refresh.",
    "visible_healthy": "No editorial flag; ranked here by model risk only.",
    "low_visibility": "Below the editorial visibility floor.",
}
q["suggested_action"] = q["archetype"].map(ACTION_BY_ARCHETYPE)
q["review_note"] = q["archetype"].map(REVIEW_NOTE_BY_ARCHETYPE)
q["check_measurement_first"] = is_top3_zero_click
q["risk_tier"] = np.select([q["lr_prob"] >= 0.65, q["lr_prob"] >= 0.55],
                           ["high", "elevated"], default="background")

# ---- reason codes: every code names its evidence (the Section 1 table) ----
codes = []
for p, peak, stale, lct10, zc, thin, big, edge, clus in zip(
        q["lr_prob"], is_peak_age, is_stale, has_low_ctr_top10, is_top3_zero_click,
        is_thin, imp >= 5000, q["threshold_edge"], q["client_cluster_top50"]):
    c = []
    if p >= 0.65:
        c.append("model_risk_high")
    elif p >= 0.55:
        c.append("model_risk_elevated")
    if peak:
        c.append("peak_risk_age_91_180")
    elif stale:
        c.append("stale_91d_plus")
    if lct10:
        c.append("low_ctr_top10")
    if zc:
        c.append("top3_zero_click_suspected")
    if thin:
        c.append("thin_data_100_499")
    if big:
        c.append("big_traffic_at_stake_5k")
    if edge:
        c.append("threshold_edge")
    if clus:
        c.append("client_cluster_top50")
    codes.append("|".join(c) if c else "model_risk_background")
q["reason_codes"] = codes

# ---- exposure (cost/value inputs; Section 2) ----
q["impressions_at_risk_if_decline"] = 0.20 * q["recent30_impressions"]
q["expected_impressions_at_risk"] = q["recent30_impressions"] * float(drop_share.median()) * q["lr_prob"]

QUEUE_COLUMNS = [
    "rank", "client_hash_id", "content_hash_id",
    "lr_prob", "risk_tier", "archetype", "suggested_action",
    "reason_codes", "review_note", "check_measurement_first",
    "recent30_impressions", "recent30_ctr_pct", "recent30_avg_position",
    "recent30_active_days", "content_age_days",
    "impressions_at_risk_if_decline", "expected_impressions_at_risk",
    "rule_score", "rule_reason_code",
]

print(f"\n{QUEUE_FOR} queue: {len(q):,} pages ranked by the refit LR (May -> June).")
print("\nRisk tiers, top 50:")
print(q.head(50)["risk_tier"].value_counts().to_string())
print("\nAction mix, top 50 (what an editor would actually work):")
print(q.head(50)["suggested_action"].value_counts().to_string())
print("\nArchetype mix, full queue:")
print(q["archetype"].value_counts().to_string())
if cluster_clients:
    print(f"\nCluster check - {len(cluster_clients)} client(s) hold >= 2x their frame share of the top 50:")
    for c in cluster_clients:
        print(f"  {c}: {int(t50_counts[c])} of top 50 (frame share {float(frame_share[c]):.1%})"
              " -> escalate to client-level review")
else:
    print("\nCluster check: no client exceeds 2x its frame share of the top 50 this cycle.")


**Pre-registered reading of the queue build** — the composition prints above; two of them get read before anything ships:

- **Risk tiers at the head.** If the top 50 is mostly `background` tier, the model is not concentrating risk where the editor works — the queue still orders, but the shipped note must say the tiers carried little separation this cycle.
- **Action mix at the head.** Monitor-class actions are free passes (no editor time); refresh-class actions are the budget. §2 turns this mix into reviewer-hours.
- The age curve above is the decay insight's second month: a peak inside 91–180d keeps the archetypes' basis (as stated in §1); a flat curve demotes `peak_risk_age_91_180` to noise and §4 fires the signal-liveness watch.

Everything else the build prints is bookkeeping: coverage of the scoring frame, dropped rows (disclosed, not hidden), and the cluster check with its base-rate-aware rule — a client must hold **2× its frame share** of the top 50 to be flagged, because the largest client's ~22% of rows makes ~11 of 50 the expectation, not a suspicious pattern.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who, what, when.** One FlyRank content strategist with a bounded refresh budget — the planning assumption throughout is **4 hours/week, one reviewer (~16 reviewer-hours/month)** — opens the queue once a month, works it top-down until budget runs out, and stops. Every row is a *suggestion for a review*, ranked; the suggested action must survive its §3 checklist before any content changes. Nothing in this notebook publishes, rewrites, redirects, or emails anything: the entire automated part ends at producing a ranked list with honest labels on it.

**The claim the queue can make** (all numbers from receipts, each precision read against its own base rate):

- Within a development month, client-grouped: the learned models ordered the review queue at or above the hand rule in every fold — means 0.512 (rule) / 0.640 (LR) / 0.740 (RF) at precision@50 against a 0.510 base rate (Week-6 receipt).
- One month out of time (March→April): head-of-queue lift mostly vanished — 0.54 against a 0.560 base rate (Week-6 receipt). Hence the one-month-instrument verdict, hence monthly refits, hence the sealed backtest in §1 grading this exact cadence on June outcomes before anything ships.
- Base rates are printed next to every precision number, always. A queue that beats 0.51 is not a queue that beats 0.56, and the word "accurate" is banned in favor of the number plus its base.

**Cost/value — where to stop working the list.** The queue's value column is `expected_impressions_at_risk` = June impressions × **measured** median drop among last month's decliners × model probability. Read it as an *ordering number for budget decisions*, not a forecast: it says where exposure concentrates, not what a refresh would recover (no causal design — §3's no-go list). Costs are planning assumptions, printed as such: measurement check ~15 min, intent review ~25 min, refresh-level review ~60–75 min. The cell below turns the top-50's actual action mix into reviewer-hours and the depth table into exposure shares — together they answer "how deep should this month's pass go" with numbers instead of vibes.

**Limits, named one by one:**

1. **Coverage floor.** Pages under 100 impressions/30d or under 14 tracked days are invisible to the queue — the funnel below quantifies the blind spot with counts, not a vague "most pages".
2. **Scored ≠ labeled population.** Labeled frames additionally require being measurable next month — Week 6 measured that at ~2.8–3.6% of candidates dropped, skewing toward pages going quiet. Everything here describes *measurable* pages.
3. **One portfolio.** ~40–44 pseudonymized clients; the largest holds ~22% of rows. Client mix moves these numbers (Week 6 corrected Week 5 on exactly this).
4. **SERP-regime blindness.** Five page-level features cannot distinguish content decay from a SERP change — an AI overview absorbing clicks reads like decay at page grain. §3's measurement checks exist because of this; a regime shift is a §4 watch.
5. **The label is a proxy.** "≥20% impressions drop in 30 days" is not "value destroyed" — seasonal pages, crawl glitches, and deliberate de-prioritization all count as decline. The model ranks a *measurement*; the review layer exists to catch the difference.
6. **Calibration is graded, not guaranteed.** Probabilities are usable for tiering only as far as the sealed calibration table supports them this cycle; if §4's calibration check fails, the queue ships ranked but the probability language goes.
7. **No content-quality features.** The model never reads the page — word count, structure, accuracy, and intent live in the human review, by design.


In [ ]:
# Coverage funnel: who the queue can and cannot see, measured on the June
# partition. Then cost/value: exposure by depth and reviewer-hours for the
# top 50 as constituted.
fact_june = f"read_parquet('{REL}/fact_content_daily_performance/{SCORE_MONTH}/*.parquet')"
funnel = con.sql(f"""
    WITH recent AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               COUNT(DISTINCT report_date) AS days
        FROM {fact_june}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*) AS pages_with_gsc_data,
        SUM(CASE WHEN days >= 14 THEN 1 ELSE 0 END) AS pages_14plus_days,
        SUM(CASE WHEN days >= 14 AND imp >= 100 THEN 1 ELSE 0 END) AS pages_eligible,
        SUM(CASE WHEN days >= 14 AND imp < 100 THEN 1 ELSE 0 END) AS pages_below_floor
    FROM recent
""").df().iloc[0]

print("June 2026 coverage funnel:")
print(f"  pages with GSC data:            {int(funnel['pages_with_gsc_data']):>9,}")
print(f"  with 14+ tracked days:          {int(funnel['pages_14plus_days']):>9,}")
print(f"  eligible (14+ days, 100+ imps): {int(funnel['pages_eligible']):>9,}")
print(f"  scored into the queue:          {len(q):>9,}  "
      f"({len(q) / int(funnel['pages_with_gsc_data']):.1%} of pages with GSC data)")
print(f"  eligible but unscored (missing position/age data): "
      f"{int(funnel['pages_eligible']) - len(q):,} - disclosed, not hidden")
print(f"  below the 100-imp floor:        {int(funnel['pages_below_floor']):>9,}  (invisible to the queue)")

total_exposure = float(q["expected_impressions_at_risk"].sum())
depth_rows = []
for depth in [20, 50, 100, 200]:
    cum = float(q.head(depth)["expected_impressions_at_risk"].sum())
    depth_rows.append({
        "queue_depth": depth,
        "expected_impressions_at_risk": int(round(cum)),
        "share_of_queue_exposure": round(cum / total_exposure, 4),
    })
depth_table = pd.DataFrame(depth_rows)
print("\nExposure by queue depth (ordering number, not a forecast):")
print(depth_table.to_string(index=False))

ASSUMED_MINUTES = {
    "investigate_measurement_first": 15,
    "review_intent_first": 25,
    "refresh_review": 60,
    "refresh_and_review_ctr": 75,
    "monitor_thin_data": 0,
    "monitor": 0,
}
top50_minutes = int(q.head(50)["suggested_action"].map(ASSUMED_MINUTES).sum())
print(f"\nTop 50 as constituted: {top50_minutes:,} assumed reviewer-minutes "
      f"= {top50_minutes / 60:.1f}h (budget assumption: 16h/month, one reviewer)")
print("Assumed minutes per action (planning assumptions, not measurements):")
for a, m in ASSUMED_MINUTES.items():
    print(f"  {a:28s} {m:>3d} min")


**Pre-registered reading:**

- **Coverage:** the funnel states the queue's blind spot as a share — everything below the floor is out of scope this month, and the paper repeats it with these counts, not a wave of the hand.
- **Depth cut:** work top-down; stop where the marginal page's expected exposure no longer clears its review cost. The depth table's saturation point is that stop, re-measured monthly. If the top-50's assumed hours overrun the 16h budget, the cut is shallower than 50 — the table decides, not enthusiasm.
- `expected_impressions_at_risk` never appears in a sentence promising recovery. It ranks. That is all it is licensed to do.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### The review checklist, per action

| action | a person checks first | assumed cost |
|---|---|---|
| `investigate_measurement_first` | Is CTR genuinely near zero — GSC data intact, zero-click SERP features, AI-overview absorption, tracking gap? Escalate to analytics if unresolved. **No content work until measurement is trusted.** | ~15 min |
| `review_intent_first` | Is the low CTR query intent (informational queries click less) or a settling page? Compare against the page's own history before spending a refresh. | ~25 min |
| `refresh_and_review_ctr` | Seasonality (a June "decline" on a seasonal page is a plateau, not decay), the page's query mix, whether a refresh is already scheduled; then title/intent + content refresh. | ~60–75 min |
| `refresh_review` | Factual decay, broken links and examples, outdated numbers; scope the refresh before starting. | ~60 min |
| `monitor_thin_data` / `monitor` | Nothing this month. Revisit when volume grows / next cycle. | 0 |

Cross-cutting checks, every cycle: **cheapest check first** (minutes before hours); **threshold-edge rows** get a second look at the underlying numbers — they sit within 10% of a cut, so a small data shift moves them; **seasonality** before calling decay; **cluster escalation** — when one client holds ≥ 2× its frame share of the top 50, the unit of work may be the template or the client, not the page.

**The action log — the one new data asset this playbook creates.** Every acted-on row gets `content_hash_id, action, action_date, actor` recorded. Not for dashboards — for the day there are enough logged refreshes to run the matched refreshed-vs-not comparison the paper's freshness claim needed (§1). The log is how "refresh" someday earns the word "recovers". Until then it is a to-do list with a memory.

### What should NOT be automated — the no-go list

1. **No auto-publishing or auto-rewriting.** The queue suggests; a human approves every content change. Quality, brand, and legal are human judgments the model cannot see.
2. **No auto-deletion, redirects, or consolidation.** Irreversible actions on pages that can look "declining" for benign reasons (season, measurement, deliberate de-prioritization) are human + business-context territory.
3. **No client-facing promises.** Never "this refresh will recover X impressions" — no causal design exists (Week 6, Finding A). Internal ordering language only.
4. **No content actions on measurement-suspect rows.** `top3_zero_click_suspected` rows get a tracking check, never a rewrite — Week 5's three confident misses were exactly this pattern.
5. **No bulk refresh from the queue alone.** A cluster flag means escalate to client-level review, not "refresh all twelve".
6. **No floor changes at runtime.** The 100 / 500 / 5,000 cuts are contract, not dials — changing them is a new experiment, which means the Week 5–6 validation cycle first.
7. **No silent refits.** Every shipped queue carries its monitoring snapshot; two consecutive head-value misses demote the model ordering to the rule, mechanically, until it re-earns the slot.
8. **No use past the month.** A July queue does not order September work. Expired queues are regenerated, not recycled.

The notebook enforces the exportable part of this list in code (the cell below): outcome columns never ship, and measurement-suspect rows never carry a content action. The rest is policy, written here so the paper inherits it verbatim.


In [ ]:
# The human-review load on this month's head, measured - plus the no-go rules
# that can be code, as code.
top50 = q.head(50)
print("Top-50 human-review load (measured on this queue):")
print(f"  measurement check first (top-3 pos, CTR < 0.30%):  {int(top50['check_measurement_first'].sum()):>3d}")
print(f"  threshold-edge rows (numbers within 10% of a cut): {int(top50['threshold_edge'].sum()):>3d}")
print(f"  client-cluster rows (>= 2x frame share of top 50): {int(top50['client_cluster_top50'].sum()):>3d}")
print(f"  thin-data rows (100-499 impressions):              {int((top50['archetype'] == 'thin_data_100_499').sum()):>3d}")
print(f"  big-traffic rows (>= 5,000 impressions):           {int((top50['recent30_impressions'] >= 5000).sum()):>3d}")

# No-go enforcement, exportable part: the queue is an action LIST, never an action TAKER.
FORBIDDEN_IN_EXPORT = ["is_declining_next30", "future30_impressions", "trend_pct", "trend_direction"]
assert not any(c in QUEUE_COLUMNS for c in FORBIDDEN_IN_EXPORT), "outcome column leaked into the export"
assert (q.loc[q["archetype"] == "top3_zero_click_suspected", "suggested_action"]
        == "investigate_measurement_first").all(), "measurement-suspect row carries a content action"
assert q["review_note"].notna().all(), "a queue row shipped without a review note"
print("\nNo-go checks: PASS - no outcome columns in the export; measurement-suspect rows")
print("carry no content action; every row carries a review note.")

print("\nTop 10 of the " + QUEUE_FOR + " queue, as an editor meets it:")
preview_cols = ["rank", "lr_prob", "risk_tier", "archetype", "suggested_action",
                "recent30_impressions", "recent30_ctr_pct", "recent30_avg_position",
                "content_age_days", "reason_codes"]
with pd.option_context("display.width", 220):
    print(q.head(10)[preview_cols].to_string(
        index=False,
        formatters={
            "lr_prob": lambda v: f"{v:.3f}",
            "recent30_ctr_pct": lambda v: f"{v:.2f}",
            "recent30_avg_position": lambda v: f"{v:.1f}",
            "recent30_impressions": lambda v: f"{int(v):,}",
        }))
print("\nA human reads this table the way Section 3 prescribes: cheapest check first,")
print("threshold-edge numbers get a second look, clusters escalate to client level.")


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Refit is the default, not the exception.** The retrain policy *is* the cadence: this notebook reruns at every month-end, refits on the freshest labeled frame, and re-scores the new month. Triggers do not decide *whether* to refit — they decide **whether the model's ordering ships or the hand rule's does**, and when to stop and investigate.

### The monthly runbook (~45 minutes)

1. Rebuild last month's labeled frame + the new scoring month (this notebook, cells unchanged; only the three month constants move).
2. Receipts: reproduce the committed row counts / base rates from last cycle's JSON — fail loudly, not silently.
3. Sealed backtest: train on the month-before-last, score last month's features, grade against last month's labels — the monitoring snapshot, same code every month.
4. Trigger table below → statuses recorded as-is into the receipt JSON.
5. No STOP: refit, score, ship queue + snapshot. Any STOP: the queue does not ship until resolved — the notebook records and discloses; the human decides.
6. Log the actions actually taken (§3's action log) — the dataset the refresh-recovery question needs.

### Triggers (thresholds fixed here, before this cycle's numbers)

| # | check | threshold | response |
|---|---|---|---|
| 1 | **Head value**: sealed-backtest LR precision@50 − base rate | ≥ +0.05 PASS · 0…+0.05 WATCH · < 0 TRIGGER | 1 TRIGGER: investigate + disclose in the shipped note. 2 consecutive: ship **rule-ordered** queue (re-sort by `rule_score` — the column already ships); model ordering returns only after one clean month. First thing to try while demoted: the two-frame training variant (§1) |
| 2 | **Base-rate regime**: new frame's decline base rate | inside [0.40, 0.65] (observed 0.51–0.56) | outside → STOP: the label's 0.80× threshold or measurement itself is moving; no queue ships until understood |
| 3 | **Frame size**: labeled rows | within ±20% of the trailing 2-frame mean | outside → STOP: pipeline/coverage problem (partition gaps, `gsc_data_available` flips) |
| 4 | **Calibration**: backtest Brier ≤ 0.25 **and** realized rate rises from bottom to top probability bin | both hold | fail → WATCH: queue may ship ranked, but drop the probability language from notes and tiers |
| 5 | **Signal liveness** (the decay insight): 91–180d decline rate vs frame base | peak above base | fail → WATCH: `peak_risk_age_91_180` and the refresh archetypes lose their basis; re-audit before prioritizing refresh work |
| 6 | **Leak canary** (re-run the Week-6 injection cell) | planted-leak AP jump ≥ +0.15 (receipt: +0.212) | fail → STOP: harness broken, nothing ships; run quarterly and after any code change |

Why two consecutive misses before demoting the model: precision@50 on 50 slots swings ~±0.15 between honest draws (Weeks 4–6 measured this repeatedly) — one bad month is noise, two is a pattern. The cost of the belt-and-braces rule is one month of a rule-ordered queue; the cost of whipsawing on noise is never knowing which instrument works.

**Light by design:** one notebook, one snapshot JSON, no dashboards, no always-on jobs. If this outgrows an hour a month, it has outgrown its evidence.


In [ ]:
# This cycle's monitoring snapshot: the trigger table above, evaluated once,
# as-is. Statuses are recorded into the receipt JSON untouched - the notebook
# never edits a status to make a cycle shippable.
lr_row = backtest.loc[backtest["scorer"] == "logistic_regression"].iloc[0]
rule_row = backtest.loc[backtest["scorer"] == "hand_rule"].iloc[0]
head_gap = float(lr_row["p50"]) - may_base

snapshot = []


def check(name, observed, rule, status):
    snapshot.append({"check": name, "observed": observed, "rule": rule, "status": status})


check("head_value_lr_p50_minus_base", round(head_gap, 4),
      ">= +0.05 PASS / 0..+0.05 WATCH / < 0 TRIGGER (2 consecutive TRIGGERs -> rule-ordered queue)",
      "PASS" if head_gap >= 0.05 else ("WATCH" if head_gap >= 0.0 else "TRIGGER"))
check("head_value_rule_p50_minus_base", round(float(rule_row["p50"]) - may_base, 4),
      "fallback price tag on the same month (context, no trigger)", "CONTEXT")

b = float(meta_may["base_rate"])
check("base_rate_in_band", round(b, 4), "inside [0.40, 0.65]",
      "PASS" if 0.40 <= b <= 0.65 else "TRIGGER")

TRAILING_2FRAME_MEAN = (96_268 + 99_279) / 2  # committed receipts: March + April frames (Week 6)
dev = abs(meta_may["rows"] - TRAILING_2FRAME_MEAN) / TRAILING_2FRAME_MEAN
check("frame_rows_within_band", int(meta_may["rows"]),
      f"within +/-20% of trailing 2-frame mean ({TRAILING_2FRAME_MEAN:,.0f})",
      "PASS" if dev <= 0.20 else "TRIGGER")

populated = calib[calib["n"] > 0]
mono = float(populated["realized_decline_rate"].iloc[-1]) >= float(populated["realized_decline_rate"].iloc[0])
calib_ok = (float(lr_row["brier"]) <= 0.25) and mono
check("calibration", f"brier {float(lr_row['brier']):.3f}, monotone {mono}",
      "brier <= 0.25 AND realized rate rises bottom->top bin", "PASS" if calib_ok else "WATCH")

peak_rate = float(age_curve.loc["91-180", "mean"]) if int(age_curve.loc["91-180", "count"]) > 0 else float("nan")
check("decay_signal_alive_91_180", round(peak_rate, 4), "91-180d decline rate > May-frame base rate",
      "PASS" if peak_rate > b else "WATCH")

check("leak_canary", "not re-run this cycle",
      "re-run the Week-6 injection cell quarterly / after code changes (receipt: +0.212 AP)", "RUNBOOK")

snapshot_df = pd.DataFrame(snapshot)
print("Monitoring snapshot, " + QUEUE_FOR + " cycle (statuses recorded as-is):")
print(snapshot_df.to_string(index=False))

n_trigger = int((snapshot_df["status"] == "TRIGGER").sum())
print(f"\nTRIGGER count: {n_trigger}. Runbook response: any TRIGGER = investigate before the queue is")
print("worked (base-rate and frame-size triggers are STOPs for shipping; head-value needs two).")
print("The notebook records and discloses; the human decides. That division is the point of this week.")


**Reading the snapshot, fixed in advance:** statuses land in the receipt JSON exactly as computed. A TRIGGER is not a crisis — it is the plan doing its job: one month of a rule-ordered queue costs approximately nothing (check 2's CONTEXT row prices the fallback on the same month), while a silently stale model costs editor-hours spent on the wrong pages. The two-consecutive rule absorbs the ±0.15 noise Weeks 4–6 measured in honest precision@50 draws without letting a real pattern run for months.

One more time, because the paper should say it too: **the notebook measures and discloses; the human decides.** Nothing in §5 ships a decision.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on these files.*

What this cycle writes, and what happens to each file:

| file | what it is | git |
|---|---|---|
| `work/outputs/action_queue_2026-07.csv` | the ranked queue: rank, probability, tier, archetype, action, reason codes, review note, the five features, exposure columns, rule score + codes | **out of git by design** (`work/**/*.csv`; the CI leak-guard blocks data files). The notebook regenerates it; the paper quotes its shape and head, never ships the file |
| `work/outputs/action_playbook_metrics.json` | the receipts: frames, sealed backtest, calibration, queue summary, coverage, cost/value, monitoring snapshot, age curve, Week-6 receipts, seed, versions | **committed** — every number the paper's recommendations section cites traces here |
| `work/figures/w07_sealed_month_lift.svg` | precision@K on the sealed month vs base rate — the honest head-of-queue picture | **committed** |
| `work/figures/w07_calibration.svg` | probability bins vs realized decline — what "risk 0.7" is actually worth | **committed** |
| `work/figures/w07_action_mix.svg` | what the queue asks for, top 50 vs full queue | **committed** |
| `work/figures/w07_exposure_by_depth.svg` | where exposure saturates — the budget cut made visible | **committed** |

(The card asks for the queue in `work/outputs/` and the reusable figures in `work/figures/` — the cell below writes exactly that split.)

Paper mapping for next week: **recommendations = §1's vocabulary + §3's no-go list, every number from the JSON**; the four figures slot into results/discussion; the queue CSV is the artifact a reviewer regenerates by running this notebook. The cell also prints a ready-to-adapt recommendations paragraph assembled from this cycle's own numbers — sentences a skeptic can check against the JSON line by line.


In [ ]:
# Exports. The queue CSV is gitignored BY DESIGN (work/**/*.csv, CI leak-guard):
# the notebook regenerates it; only the receipt JSON and figures are committed.
# Paths are relative to work/notebooks/, like Weeks 4-6.
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})

OUTPUT_DIR = Path("../../work/outputs")
FIG_DIR = Path("../../work/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

queue_path = OUTPUT_DIR / f"action_queue_{QUEUE_FOR}.csv"
q[QUEUE_COLUMNS].to_csv(queue_path, index=False)

# The shipped note - self-updating disclosure generated from this cycle's snapshot.
head_status = next(s for s in snapshot if s["check"] == "head_value_lr_p50_minus_base")
shipped_note = (
    f"{QUEUE_FOR} queue, ordered by logistic regression refit on May->June labels. "
    f"Head support this cycle: sealed-month precision@50 {float(lr_row['p50']):.3f} against a "
    f"{may_base:.3f} base rate ({head_status['status']}). Decision-support only; every row passes "
    f"the Section-3 human checklist before any action."
)
print(shipped_note)


def r4(x):
    return round(float(x), 4)


metrics = {
    "notebook": "w07_action_playbook",
    "queue_for": QUEUE_FOR,
    "ship_scorer": "logistic_regression",
    "ship_choice_basis": (
        "pre-registered from Week-6 out-of-time receipts: LR matched or beat RF at precision@50 "
        "in all five out-of-time draws with the better Brier; the sealed backtest measures the "
        "cadence, it does not select the model"
    ),
    "cadence": {
        "backtest_train": BACKTEST_TRAIN_MONTH,
        "ship_train": SHIP_TRAIN_MONTH,
        "scored": SCORE_MONTH,
        "valid_for": QUEUE_FOR,
    },
    "frames": {"april": meta_april, "may": meta_may, "june_scoring": meta_june},
    "sealed_month_backtest": {
        "train": "april frame (Apr features -> May labels)",
        "test": "may frame (May features, June outcomes - sealed month, first touch)",
        "base_rate": r4(may_base),
        "metrics": {row["scorer"]: {k: r4(row[k]) for k in
                                    ["p20", "p50", "ndcg50", "ap", "roc_auc", "brier"]}
                    for _, row in backtest.iterrows()},
        "calibration_lr": calib[calib["n"] > 0].to_dict(orient="records"),
    },
    "measured_decline_drop_may_frame": {
        "n_decliners": int(len(drop_share)),
        "median": r4(drop_share.median()),
        "mean": r4(drop_share.mean()),
    },
    "age_curve_may_frame": {str(band): {"n": int(row["count"]), "decline_rate": r4(row["mean"])}
                            for band, row in age_curve.iterrows()},
    "queue_summary": {
        "rows": int(len(q)),
        "risk_tier_counts": {k: int(v) for k, v in q["risk_tier"].value_counts().items()},
        "action_counts_top50": {k: int(v) for k, v in
                                q.head(50)["suggested_action"].value_counts().items()},
        "archetype_counts": {k: int(v) for k, v in q["archetype"].value_counts().items()},
        "top50_measurement_checks": int(q.head(50)["check_measurement_first"].sum()),
        "top50_cluster_rows": int(q.head(50)["client_cluster_top50"].sum()),
        "top50_threshold_edge_rows": int(q.head(50)["threshold_edge"].sum()),
    },
    "coverage_june": {k: int(v) for k, v in funnel.items()},
    "cost_value": {
        "expected_exposure_total": int(round(total_exposure)),
        "depth_table": depth_table.to_dict(orient="records"),
        "assumed_minutes_per_action": ASSUMED_MINUTES,
        "top50_assumed_hours": round(top50_minutes / 60.0, 1),
        "assumption_note": (
            "minutes are planning assumptions, not measurements; exposure is an ordering "
            "number, not a forecast"
        ),
    },
    "monitoring": {
        "thresholds": {
            "head_value": ">= +0.05 PASS / 0..+0.05 WATCH / < 0 TRIGGER; 2 consecutive TRIGGERs -> rule-ordered queue",
            "base_rate_band": [0.40, 0.65],
            "frame_rows_band": "within +/-20% of trailing 2-frame mean",
            "calibration": "brier <= 0.25 and realized rate rises bottom->top bin",
            "signal_liveness": "91-180d decline rate > frame base rate",
            "leak_canary": "planted-leak AP jump >= +0.15 (Week-6 receipt +0.212), quarterly",
        },
        "snapshot": snapshot,
        "shipped_note": shipped_note,
    },
    "receipts_from_w6": W6,
    "seed": SEED,
    "library_versions": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "duckdb": duckdb.__version__,
    },
    "generated_run_date": date.today().isoformat(),
}
metrics_path = OUTPUT_DIR / "action_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, default=str))

# ---- Figure 1: sealed-month lift - the honest head-of-queue picture ----
ks = [10, 20, 50, 100, 200]
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, label in [("logistic_regression", "logistic regression (ships)"),
                    ("random_forest", "random forest"),
                    ("hand_rule", "hand rule (fallback order)")]:
    ax.plot(ks, [precision_at_k(y_test, scorer_preds[name], k) for k in ks],
            marker="o", label=label)
ax.axhline(may_base, ls="--", lw=1, color="gray")
ax.annotate(f"base rate {may_base:.3f}", (200, may_base), textcoords="offset points",
            xytext=(-4, 4), ha="right", fontsize=8, color="gray")
ax.set_xscale("log")
ax.set_xticks(ks)
ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel("queue depth K")
ax.set_ylabel("precision@K")
ax.set_title("Sealed-month backtest: train April frame, score May features\n"
             "(outcomes June 2026; one draw, no tuning; read against the base rate)")
ax.legend(frameon=False, fontsize=8, loc="lower left")
fig.savefig(FIG_DIR / "w07_sealed_month_lift.svg")
plt.show()

# ---- Figure 2: calibration - what 'risk 0.7' is worth ----
c_cal = calib[calib["n"] > 0].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(c_cal["prob_bin"], c_cal["realized_decline_rate"], color="#4E79A7",
       label="realized decline rate")
ax.plot(c_cal["prob_bin"], c_cal["mean_pred"], marker="D", ms=5, ls="none",
        color="#E15759", label="mean predicted probability")
for i, n in enumerate(c_cal["n"]):
    ax.annotate(f"n={n:,}", (i, 0.03), ha="center", fontsize=7, color="white")
ax.axhline(may_base, ls="--", lw=1, color="gray")
ax.set_ylim(0, 1)
ax.set_xlabel("LR probability bin (cuts fixed before looking)")
ax.set_ylabel("decline rate")
ax.set_title("Calibration on the sealed month\n('risk 0.70' is worth what the >=0.70 bar shows)")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / "w07_calibration.svg")
plt.show()

# ---- Figure 3: action mix, top 50 vs full queue ----
ACTION_ORDER = ["refresh_and_review_ctr", "refresh_review", "review_intent_first",
                "investigate_measurement_first", "monitor_thin_data", "monitor"]
counts_top = q.head(50)["suggested_action"].value_counts().reindex(ACTION_ORDER, fill_value=0)
counts_all = q["suggested_action"].value_counts().reindex(ACTION_ORDER, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
ylabels = ACTION_ORDER[::-1]
axes[0].barh(ylabels, counts_top.values[::-1], color="#426B69")
axes[0].set_title("Top 50 (what gets worked)")
axes[1].barh(ylabels, counts_all.values[::-1], color="#6F4E7C")
axes[1].set_title("Full queue (who is on it)")
for ax_ in axes:
    ax_.set_xlabel("pages")
fig.suptitle(QUEUE_FOR + " queue: suggested actions", y=1.02, fontsize=11)
fig.savefig(FIG_DIR / "w07_action_mix.svg")
plt.show()

# ---- Figure 4: where exposure saturates - the budget cut ----
depths = np.arange(1, 501)
cum_share = (q.head(500)["expected_impressions_at_risk"].cumsum() / total_exposure).to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(depths, cum_share, color="#4E79A7")
for d in [20, 50, 100]:
    ax.axvline(d, ls=":", lw=1, color="gray")
    ax.annotate(f"{d}: {cum_share[d - 1]:.0%}", (d, cum_share[d - 1]),
                textcoords="offset points", xytext=(5, -12), fontsize=8)
ax.set_ylim(0, 1)
ax.set_xlim(1, 500)
ax.set_xlabel("queue depth (top N pages)")
ax.set_ylabel("share of total expected exposure")
ax.set_title("Where exposure saturates: cumulative expected impressions at risk\n"
             "(ordering number for budget cuts, not a forecast)")
fig.savefig(FIG_DIR / "w07_exposure_by_depth.svg")
plt.show()

# ---- A recommendations paragraph the paper can adapt, assembled from this
# cycle's own numbers (every clause checks against the JSON above) ----
share50 = float(depth_table.loc[depth_table["queue_depth"] == 50,
                                "share_of_queue_exposure"].iloc[0])
paper_paragraph = (
    f"**Recommendations ({QUEUE_FOR} queue).** Each month the refit logistic model ranks the pages "
    f"measurable enough to review - {len(q):,} this cycle - and every row carries an action, reason "
    f"codes, a human-review checklist, and an exposure estimate. The shipping cadence was graded once "
    f"on the sealed June outcome month: head-of-queue precision@50 {float(lr_row['p50']):.3f} against "
    f"a {may_base:.3f} base rate ({head_status['status']}; one draw, directional). Under assumed review "
    f"costs the top 50 rows cost about {top50_minutes / 60:.0f} reviewer-hours and hold {share50:.0%} "
    f"of the queue's expected exposure, so the monthly budget cut is a measured decision. Nothing is "
    f"automated: refresh actions claim review priority, never recovery, and the no-go list (no "
    f"auto-publishing, no irreversible actions, no client-facing promises) travels with the queue."
)
print(paper_paragraph)

print("\nExports written:")
print(f"  {queue_path}  ({queue_path.stat().st_size / 1e6:.1f} MB) - ranked queue; gitignored BY DESIGN, regenerates")
print(f"  {metrics_path} - receipt JSON; COMMIT")
for f in sorted(FIG_DIR.glob("w07_*.svg")):
    print(f"  {f} - figure; COMMIT to work/figures/")

gi = Path("../../.gitignore")
if gi.exists():
    assert "work/**/*.csv" in gi.read_text(), "expected the work/**/*.csv guard in .gitignore"
    print("\n.gitignore receipt: 'work/**/*.csv' present - the queue CSV cannot be committed by accident.")
print("\nAfter the run: download action_playbook_metrics.json and the four w07_*.svg figures from")
print("Colab's file pane into the repo (work/outputs/ and work/figures/), commit them with the")
print("notebook, and leave the queue CSV out of git - the notebook is its regeneration.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all, executed in Colab and saved)
- [x] No client names, URLs, or private queries anywhere — pseudonymous hash IDs only
- [x] My claims use careful words: observed, measured, directional, decision-support — no causal recovery claims, every precision next to its base rate
- [x] Exports written: queue CSV (regenerated each run, never committed), receipt JSON + four figures (committed)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
